In [ ]:
import pandas as pd

df_bolsa_jan = pd.read_csv("202501_NovoBolsaFamilia.csv", sep=';', encoding='latin1')
df_bolsa_fev = pd.read_csv("202502_NovoBolsaFamilia.csv", sep=';', encoding='latin1') 


FileNotFoundError: [Errno 2] No such file or directory: '202501_NovoBolsaFamilia.csv'

In [4]:
df_bolsa = pd.concat([df_bolsa_jan, df_bolsa_fev], ignore_index=True)

df_bolsa.shape

df_bolsa.tail(10)

NameError: name 'df_bolsa_jan' is not defined

In [5]:
df_bolsa.to_parquet("bolsafamilia_202501_202502.parquet", index=False)

NameError: name 'df_bolsa' is not defined

In [2]:
import pandas as pd
import polars as pl

df_bolsapl = pl.read_parquet("bolsafamilia_202501_202502.parquet")

In [3]:
{
    "Shape": df_bolsapl.shape,
    "Head": df_bolsapl.head(10),
    "Tail": df_bolsapl.tail(10)
}


{'Shape': (40810993, 9),
 'Head': shape: (10, 9)
 ┌────────────┬────────────┬─────┬────────────┬───┬────────────┬────────────┬───────────┬───────────┐
 │ MÊS COMPET ┆ MÊS        ┆ UF  ┆ CÓDIGO     ┆ … ┆ CPF        ┆ NIS        ┆ NOME FAVO ┆ VALOR     │
 │ ÊNCIA      ┆ REFERÊNCIA ┆ --- ┆ MUNICÍPIO  ┆   ┆ FAVORECIDO ┆ FAVORECIDO ┆ RECIDO    ┆ PARCELA   │
 │ ---        ┆ ---        ┆ str ┆ SIAFI      ┆   ┆ ---        ┆ ---        ┆ ---       ┆ ---       │
 │ i64        ┆ i64        ┆     ┆ ---        ┆   ┆ str        ┆ f64        ┆ str       ┆ str       │
 │            ┆            ┆     ┆ i64        ┆   ┆            ┆            ┆           ┆           │
 ╞════════════╪════════════╪═════╪════════════╪═══╪════════════╪════════════╪═══════════╪═══════════╡
 │ 202501     ┆ 202308     ┆ SP  ┆ 7071       ┆ … ┆ ***.085.10 ┆ 2.0644e10  ┆ FERNANDA  ┆ 650,00    │
 │            ┆            ┆     ┆            ┆   ┆ 6-**       ┆            ┆ RAMOS     ┆           │
 │            ┆            ┆     

In [13]:
df_bolsapl.columns


['MÊS COMPETÊNCIA',
 'MÊS REFERÊNCIA',
 'UF',
 'CÓDIGO MUNICÍPIO SIAFI',
 'NOME MUNICÍPIO',
 'CPF FAVORECIDO',
 'NIS FAVORECIDO',
 'NOME FAVORECIDO',
 'VALOR PARCELA']

In [14]:
#conversão dos nomes das colunas:

df_bolsapl = df_bolsapl.rename({
    "MÊS COMPETÊNCIA": "mes_competencia",
    "MÊS REFERÊNCIA": "mes_referencia",
    "UF": "uf",
    "CÓDIGO MUNICÍPIO SIAFI": "cod_municipio_siafi",
    "NOME MUNICÍPIO": "nome_municipio",
    "CPF FAVORECIDO": "cpf_favorecido",
    "NIS FAVORECIDO": "nis_favorecido",
    "NOME FAVORECIDO": "nome_favorecido",
    "VALOR PARCELA": "valor_parcela"
})


In [39]:
df_bolsapl.head(10)

mes_competencia,mes_referencia,uf,cod_municipio_siafi,nome_municipio,nis_favorecido,nome_favorecido,valor_parcela
i64,i64,str,i64,str,str,str,f64
202501,202308,"""SP""",7071,"""santos""","""20643890445""","""FERNANDA RAMOS TEIXEIRA""",650.0
202501,202309,"""SP""",7071,"""santos""","""20643890445""","""FERNANDA RAMOS TEIXEIRA""",650.0
202501,202310,"""SP""",7071,"""santos""","""20643890445""","""FERNANDA RAMOS TEIXEIRA""",650.0
202501,202311,"""SP""",7071,"""santos""","""20643890445""","""FERNANDA RAMOS TEIXEIRA""",650.0
202501,202312,"""SP""",7071,"""santos""","""20643890445""","""FERNANDA RAMOS TEIXEIRA""",650.0
202501,202401,"""CE""",1389,"""fortaleza""","""13024055192""","""IRENICE CAMELO VALETE""",750.0
202501,202401,"""PA""",479,"""limoeiro do ajuru""","""16025132403""","""ANTONIA ELIZANA DA SILVA""",600.0
202501,202401,"""SP""",6313,"""carapicuiba""","""13310239852""","""DAMARIS DOS SANTOS OLIVEIRA""",800.0
202501,202401,"""SP""",6313,"""carapicuiba""","""20217442743""","""PRISCILA HELENA DE LIMA""",650.0


In [16]:
#conversão da coluna de valor para numero com . ao invés de ,

df_bolsapl = df_bolsapl.with_columns([
    pl.col("valor_parcela")
    .str.replace(",", ".")
    .cast(pl.Float64)
    .alias("valor_parcela")
])

df_bolsapl.select("valor_parcela").head(10)

valor_parcela
f64
650.0
650.0
650.0
650.0
650.0
750.0
600.0
800.0
650.0


In [18]:
df_bolsapl.null_count()


mes_competencia,mes_referencia,uf,cod_municipio_siafi,nome_municipio,cpf_favorecido,nis_favorecido,nome_favorecido,valor_parcela
u32,u32,u32,u32,u32,u32,u32,u32,u32
0,0,0,0,0,7942467,725,0,0


In [19]:
#descarte da coluna CPF que tem muitos valores nulos

df_bolsapl = df_bolsapl.drop("cpf_favorecido")


In [21]:
#comparação das colunas de mês que podem estar redundantes:

(df_bolsapl.select([
    (pl.col("mes_competencia") != pl.col("mes_referencia")).alias("diferente")
])
.filter(pl.col("diferente") == True)
).height


193687

In [22]:
#normalizar nomes retirando acentos e espaços para não atrapalhar análises

import unidecode

# função auxiliar para remover acentos:

def normalizar_texto(s):
    if s is not None:
        return unidecode.unidecode(s.strip().lower())
    return s

# aplicar no nome do município:

df_bolsapl = df_bolsapl.with_columns([
    pl.col("nome_municipio").map_elements(normalizar_texto).alias("nome_municipio")
])


C:\Users\Win\AppData\Local\Temp\ipykernel_10464\3950830719.py:14: MapWithoutReturnDtypeWarning: Calling `map_elements` without specifying `return_dtype` can lead to unpredictable results. Specify `return_dtype` to silence this warning.
  df_bolsapl = df_bolsapl.with_columns([


In [24]:
#garante que a coluna esteja no formato str ao invés de f64: 

df_bolsapl = df_bolsapl.with_columns([
    pl.col("nis_favorecido").cast(pl.Utf8).alias("nis_favorecido")
])

In [25]:
#verificando se existe algum valor não inteiro (que não seja x.0) na coluna:

df_bolsapl.select(
    (pl.col("nis_favorecido")
    .cast(pl.Float64) % 1 != 0)
    .sum()
    .alias("nis_nao_inteiros")
)


nis_nao_inteiros
u32
0


In [26]:
df_bolsapl = df_bolsapl.with_columns([
    pl.col("nis_favorecido")
    .cast(pl.Utf8)
    .str.replace(r"\.0$", "")
    .alias("nis_favorecido")
])

In [ ]:
#verificar se existem duplicatas no df:

num_duplicadas = df_bolsapl.shape[0] - df_bolsapl.unique().shape[0]
print(f"Linhas duplicadas: {num_duplicadas}")

Linhas duplicadas: 35


In [6]:
duplicadas = (
    df_bolsapl
    .group_by(df_bolsapl.columns)
    .agg(pl.count().alias("repeticoes"))
    .filter(pl.col("repeticoes") > 1)
)

print(duplicadas)


C:\Users\Win\AppData\Local\Temp\ipykernel_21748\2879834304.py:4: DeprecationWarning: `pl.count()` is deprecated. Please use `pl.len()` instead.
(Deprecated in version 0.20.5)
  .agg(pl.count().alias("repeticoes"))


shape: (24, 10)
┌────────────┬────────────┬─────┬────────────┬───┬────────────┬────────────┬───────────┬───────────┐
│ MÊS COMPET ┆ MÊS        ┆ UF  ┆ CÓDIGO     ┆ … ┆ NIS        ┆ NOME       ┆ VALOR     ┆ repeticoe │
│ ÊNCIA      ┆ REFERÊNCIA ┆ --- ┆ MUNICÍPIO  ┆   ┆ FAVORECIDO ┆ FAVORECIDO ┆ PARCELA   ┆ s         │
│ ---        ┆ ---        ┆ str ┆ SIAFI      ┆   ┆ ---        ┆ ---        ┆ ---       ┆ ---       │
│ i64        ┆ i64        ┆     ┆ ---        ┆   ┆ f64        ┆ str        ┆ str       ┆ u32       │
│            ┆            ┆     ┆ i64        ┆   ┆            ┆            ┆           ┆           │
╞════════════╪════════════╪═════╪════════════╪═══╪════════════╪════════════╪═══════════╪═══════════╡
│ 202502     ┆ 202502     ┆ SP  ┆ 7107       ┆ … ┆ null       ┆ *** BENEFI ┆ 750,00    ┆ 4         │
│            ┆            ┆     ┆            ┆   ┆            ┆ CIÁRIO     ┆           ┆           │
│            ┆            ┆     ┆            ┆   ┆            ┆ MENOR DE   

In [7]:
duplicadas_todas = df_bolsapl.filter(df_bolsapl.is_duplicated(keep=False))
print(duplicadas_todas)


TypeError: DataFrame.is_duplicated() got an unexpected keyword argument 'keep'